# Advanced Streaming Fault Detection

This notebook covers advanced streaming features including:
- Data buffering and windowing
- Online learning
- Custom data sources
- Performance optimization
- Result analysis

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os
from datetime import datetime

# Import EnergyFaultDetector components
from energy_fault_detector import StreamingFaultDetector, Config
from energy_fault_detector.data_sources import CSVStreamDataSource, SimulatedFaultDataSource, StreamConfig
from energy_fault_detector.streaming import DataBuffer, SlidingWindowBuffer, StreamingResult
from energy_fault_detector.config import generate_quickstart_config

print("✓ All imports successful")

## 1. Data Buffering for Sequence Models

Learn how to use data buffers and sliding windows for sequence-based models.

In [ ]:
# Create a DataBuffer
buffer = DataBuffer(max_size=100)

# Add some data
data = np.random.randn(50, 3)
buffer.add(data)

print(f"Buffer size: {buffer.size}")
print(f"Buffer is full: {buffer.is_full}")
print(f"Buffer data shape: {buffer.data.shape}")

# Add more data to fill the buffer
buffer.add(np.random.randn(60, 3))
print(f"\nAfter adding more data:")
print(f"Buffer size: {buffer.size}")
print(f"Buffer is full: {buffer.is_full}")

# Get recent data
recent = buffer.get_recent(20)
print(f"Recent data shape: {recent.shape}")

# Convert to DataFrame
df = buffer.to_dataframe(columns=['sensor_1', 'sensor_2', 'sensor_3'])
print(f"DataFrame shape: {df.shape}")
print(df.head())

In [ ]:
# Create a SlidingWindowBuffer
window_buffer = SlidingWindowBuffer(
    window_size=10,  # Each window has 10 timesteps
    stride=2,        # Step 2 samples between windows
    max_windows=50   # Store maximum 50 windows
)

# Add data
data = np.random.randn(30, 2)  # 30 samples, 2 features
new_windows = window_buffer.add(data)

print(f"Number of new windows created: {len(new_windows)}")
print(f"Total windows in buffer: {window_buffer.n_windows}")
print(f"Buffer size: {window_buffer.buffer_size}")

# Get sequence dataset (for LSTM/CNN models)
X, timestamps = window_buffer.get_sequence_dataset()
print(f"\nSequence dataset shape: {X.shape}")
print(f"  - Number of windows: {X.shape[0]}")
print(f"  - Window size: {X.shape[1]}")
print(f"  - Number of features: {X.shape[2]}")

# Get as DataFrame
window_df = window_buffer.get_window_dataframe()
print(f"Window DataFrame shape: {window_df.shape}")

## 2. Streaming with Window Buffer

Use a StreamingFaultDetector with a window buffer for sequence data.

In [ ]:
# Create synthetic sequential data
np.random.seed(42)
n_samples = 1000
time = np.linspace(0, 20 * np.pi, n_samples)

# Create sequential patterns
data = {
    'value': np.sin(time) + np.random.randn(n_samples) * 0.1,
    'trend': time / 10,
}

# Add sequential faults
for i in range(200, 250):
    data['value'][i] += 3.0

df = pd.DataFrame(data)

# Save to CSV
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    df.to_csv(f.name, index=False)
    csv_path = f.name

print(f"Created sequential data with {n_samples} samples")

In [ ]:
# Create streaming detector with window buffer
config = generate_quickstart_config()

detector = StreamingFaultDetector(
    config=config,
    buffer_size=200,
    window_size=20,  # Use window buffer for sequence data
    online_learning=False
)

# Initialize with normal data
normal_data = df.iloc[:300]
detector.initialize(fit_data=normal_data)

# Create stream source
source = CSVStreamDataSource(
    file_path=csv_path,
    config=StreamConfig(batch_size=50, delay_seconds=0)
)

# Process stream
print("Processing sequential data stream...")
result = detector.process_stream(source)

print(f"\nResults:")
print(f"  Total samples: {result.total_samples}")
print(f"  Anomalies detected: {result.total_anomalies}")
print(f"  Throughput: {result.throughput:.2f} samples/sec")

## 3. Online Learning

Demonstrate online learning with streaming data.

In [ ]:
# Create a data source with changing patterns (concept drift)
class DriftingDataSource:
    """Data source that gradually changes its pattern to simulate concept drift."""
    
    def __init__(self, n_samples=1000, n_features=2):
        self.n_samples = n_samples
        self.n_features = n_features
        self.current_sample = 0
        self.batch_index = 0
        
    def __iter__(self):
        self.current_sample = 0
        self.batch_index = 0
        return self
    
    def __next__(self):
        from energy_fault_detector.data_sources import DataBatch
        
        if self.current_sample >= self.n_samples:
            raise StopIteration
        
        # Generate batch with changing mean
        batch_size = 100
        end_sample = min(self.current_sample + batch_size, self.n_samples)
        
        # Mean shifts over time (concept drift)
        drift = (self.current_sample / self.n_samples) * 5.0
        
        data = np.random.randn(batch_size, self.n_features) + drift
        timestamps = np.arange(self.current_sample, end_sample)
        
        batch = DataBatch(
            data=pd.DataFrame(data, columns=[f'feature_{i}' for i in range(self.n_features)]),
            timestamps=timestamps,
            batch_index=self.batch_index,
            is_complete=end_sample >= self.n_samples
        )
        
        self.current_sample = end_sample
        self.batch_index += 1
        
        return batch

# Create drifting data source
drifting_source = DriftingDataSource(n_samples=1000, n_features=2)

# Create detector with online learning enabled
detector_online = StreamingFaultDetector(
    config=config,
    buffer_size=300,
    online_learning=True,
    online_learning_interval=5  # Update every 5 batches
)

# Initialize with initial normal data
initial_data = pd.DataFrame(np.random.randn(200, 2), columns=['feature_0', 'feature_1'])
detector_online.initialize(fit_data=initial_data)

print("✓ Online learning detector initialized")

In [ ]:
# Process the drifting stream
print("Processing stream with concept drift and online learning...")
online_result = detector_online.process_stream(drifting_source)

print(f"\nOnline Learning Results:")
print(f"  Total samples: {online_result.total_samples}")
print(f"  Anomalies detected: {online_result.total_anomalies}")
print(f"  Batches processed: {len(online_result.batch_results)}")

# Plot anomaly scores over time
plt.figure(figsize=(12, 6))
plt.plot(online_result.all_scores, label='Anomaly Score', alpha=0.7)
plt.axhline(1.0, color='r', linestyle='--', label='Threshold (approx)')
plt.title('Anomaly Scores with Online Learning')
plt.xlabel('Sample Index')
plt.ylabel('Anomaly Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Performance Analysis

Analyze the performance of streaming fault detection.

In [ ]:
# Compare performance with different batch sizes
batch_sizes = [10, 50, 100, 200]
performance_results = []

for batch_size in batch_sizes:
    # Create source with this batch size
    source = CSVStreamDataSource(
        file_path=csv_path,
        config=StreamConfig(batch_size=batch_size, delay_seconds=0)
    )
    
    # Create detector
    detector_perf = StreamingFaultDetector(config=config)
    detector_perf.initialize(fit_data=df.iloc[:300])
    
    # Process stream
    start_time = datetime.now()
    result = detector_perf.process_stream(source)
    end_time = datetime.now()
    
    total_time = (end_time - start_time).total_seconds()
    performance_results.append({
        'batch_size': batch_size,
        'total_samples': result.total_samples,
        'total_time': total_time,
        'throughput': result.throughput,
        'avg_processing_time': result.avg_processing_time,
        'n_batches': len(result.batch_results)
    })
    
    detector_perf.close()

# Display performance results
perf_df = pd.DataFrame(performance_results)
print("Performance Comparison:\n")
print(perf_df.to_string(index=False))

# Plot throughput vs batch size
plt.figure(figsize=(10, 6))
plt.bar(perf_df['batch_size'], perf_df['throughput'])
plt.xlabel('Batch Size')
plt.ylabel('Throughput (samples/sec)')
plt.title('Throughput vs Batch Size')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Custom Data Source

Create and use a custom data source.

In [ ]:
from energy_fault_detector.data_sources import DataSource, DataBatch, StreamConfig
from typing import Iterator

class CustomRandomWalkSource(DataSource):
    """Custom data source that generates random walk data."""
    
    def __init__(self, n_samples=500, n_features=1, config=None):
        super().__init__(config=config or StreamConfig(batch_size=50))
        self.n_samples = n_samples
        self.n_features = n_features
        self.current_sample = 0
        self._data = None
        
    def open(self):
        # Generate random walk data
        np.random.seed(42)
        self._data = np.cumsum(np.random.randn(self.n_samples, self.n_features), axis=0)
        self.current_sample = 0
        self._batch_index = 0
        self._is_open = True
        return self
    
    def close(self):
        self._data = None
        self.current_sample = 0
        self._is_open = False
    
    def reset(self):
        self.current_sample = 0
        self._batch_index = 0
    
    def __next__(self):
        if not self._is_open:
            raise RuntimeError("Data source is not open")
        
        if self.current_sample >= self.n_samples:
            raise StopIteration
        
        batch_size = self.config.batch_size
        end_sample = min(self.current_sample + batch_size, self.n_samples)
        
        batch_data = self._data[self.current_sample:end_sample]
        timestamps = np.arange(self.current_sample, end_sample)
        
        batch = DataBatch(
            data=pd.DataFrame(batch_data, columns=[f'walk_{i}' for i in range(self.n_features)]),
            timestamps=timestamps,
            batch_index=self._batch_index,
            is_complete=end_sample >= self.n_samples
        )
        
        self.current_sample = end_sample
        self._batch_index += 1
        
        return batch

# Use the custom data source
custom_source = CustomRandomWalkSource(
    n_samples=500,
    n_features=2,
    config=StreamConfig(batch_size=50, delay_seconds=0)
)

# Create detector
detector_custom = StreamingFaultDetector(config=config)

# Initialize with some normal random walk data
normal_walk = pd.DataFrame(np.cumsum(np.random.randn(200, 2), axis=0), 
                          columns=['walk_0', 'walk_1'])
detector_custom.initialize(fit_data=normal_walk)

# Process custom stream
print("Processing custom random walk stream...")
custom_result = detector_custom.process_stream(custom_source)

print(f"\nCustom Source Results:")
print(f"  Total samples: {custom_result.total_samples}")
print(f"  Anomalies detected: {custom_result.total_anomalies}")

## 6. Result Analysis and Visualization

Advanced analysis of streaming results.

In [ ]:
# Get all results from the first detector
all_predictions = result.all_predictions
all_scores = result.all_scores

# Calculate statistics
print("Result Statistics:")
print(f"  Mean anomaly score: {np.mean(all_scores):.4f}")
print(f"  Std anomaly score: {np.std(all_scores):.4f}")
print(f"  Max anomaly score: {np.max(all_scores):.4f}")
print(f"  Min anomaly score: {np.min(all_scores):.4f}")

# Plot histogram of anomaly scores
plt.figure(figsize=(12, 6))
plt.hist(all_scores, bins=50, alpha=0.7, label='Anomaly Scores')
plt.axvline(np.mean(all_scores), color='r', linestyle='--', label='Mean')
plt.axvline(np.percentile(all_scores, 95), color='g', linestyle='--', label='95th Percentile')
plt.title('Distribution of Anomaly Scores')
plt.xlabel('Anomaly Score')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot processing times
processing_times = result.processing_times

plt.figure(figsize=(12, 6))
plt.plot(processing_times, marker='o', linestyle='-', alpha=0.7)
plt.axhline(np.mean(processing_times), color='r', linestyle='--', label='Mean')
plt.title('Processing Time per Batch')
plt.xlabel('Batch Index')
plt.ylabel('Processing Time (seconds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Processing time statistics:")
print(f"  Mean: {np.mean(processing_times):.6f}s")
print(f"  Std: {np.std(processing_times):.6f}s")
print(f"  Min: {np.min(processing_times):.6f}s")
print(f"  Max: {np.max(processing_times):.6f}s")

## 7. Cleanup

Clean up resources.

In [ ]:
# Clean up temporary files
os.unlink(csv_path)
print("✓ Temporary file cleaned up")

# Close all detectors
detector.close()
detector_online.close()
detector_custom.close()
print("✓ All detectors closed")

print("\n🎉 Advanced Streaming Fault Detection Complete!")